# 🎋 BỐC QUẺ ĐẦU NĂM — TIKTOK LIVE INTERACTIVE 9:16
### HỆ THỐNG PHÁT SÓNG TƯƠNG TÁC XIN XĂM - RÚT QUẺ ĐẦU XUÂN TRÊN GOOGLE COLAB

Hệ thống phát sóng tương tác chuẩn khung hình dọc **9:16 (1080x1920)** với phong cách Tết cổ truyền rực rỡ, hỗ trợ:
- **Cách 1 (Khuyên dùng - Nhẹ máy nhất):** Colab làm máy chủ -> Cấp link HTTPS Cloudflare Tunnel để dán vào mục **Liên kết (Browser Source)** trong **TikTok LIVE Studio** hoặc **OBS Studio** (Không bị hỏi đăng nhập, không lỗi màn hình đen).
- **Cách 2 (Tự động 24/7):** Colab tự tạo Màn hình ảo (Xvfb 1080x1920) và dùng FFmpeg đẩy luồng trực tiếp lên TikTok bằng **Stream Key** mà không cần bật máy tính cá nhân!

In [ ]:
# @title 🚀 BƯỚC 1: Khởi Chạy Server Bốc Quẻ & Lấy Link Dán Vào TikTok LIVE Studio / OBS { display-mode: "form" }
# @markdown Bấm nút **Play ▶** bên trái để khởi động toàn bộ hệ thống:

TikTok_Username = "@vuatiengviet171095" # @param {type:"string"}

import os, sys, time, re, subprocess
from IPython.display import display, HTML

print("⏳ [1/4] Đang đồng bộ mã nguồn Game Bốc Quẻ Đầu Năm...")
!git clone https://github.com/tuna1710/tiktok-live-game-skill.git /content/app 2>/dev/null || (cd /content/app && git fetch origin && git reset --hard origin/main && git pull)
%cd /content/app/games/boc-que-dau-nam

print("⏳ [2/4] Đang kiểm tra thư viện Node.js & Cloudflare Tunnel...")
!npm install --silent
![ ! -f /usr/local/bin/cloudflared ] && wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

print("⏳ [3/4] Đang dọn dẹp tiến trình cũ và khởi chạy Server Game...")
!pkill -9 -f "server.js" > /dev/null 2>&1
!pkill -9 -f "cloudflared" > /dev/null 2>&1
time.sleep(1)

# Thiết lập biến môi trường
clean_username = TikTok_Username.strip()
env = os.environ.copy()
env["PORT"] = "3005"
env["TIKTOK_USERNAME"] = clean_username

# Khởi chạy server.js ở chế độ nền
with open("/tmp/game_server.log", "w") as f_log:
    server_proc = subprocess.Popen(["node", "server.js"], cwd="/content/app/games/boc-que-dau-nam", env=env, stdout=f_log, stderr=subprocess.STDOUT)
time.sleep(3)

# Khởi chạy Cloudflare Tunnel công khai
print("⏳ [4/4] Đang khởi tạo đường truyền HTTPS công khai qua Cloudflare Tunnel...")
with open("/tmp/cloudflared.log", "w") as cf_log:
    tunnel_proc = subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:3005"], stdout=cf_log, stderr=subprocess.STDOUT)

public_url = None
for _ in range(15):
    time.sleep(1)
    if os.path.exists("/tmp/cloudflared.log"):
        with open("/tmp/cloudflared.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    obs_link = f"{public_url}/index.html"
    admin_link = f"{public_url}/admin.html"
    html_card = f"""
    <div style="background: linear-gradient(135deg, #4a0000 0%, #1f0000 100%); border: 2px solid #e5a93c; border-radius: 16px; padding: 22px; color: #fffaf0; font-family: sans-serif; box-shadow: 0 8px 30px rgba(229,169,60,0.3);">
        <h2 style="color: #ffd700; margin-top: 0; font-size: 22px;">🎋 GAME BỐC QUẺ ĐẦU NĂM ĐÃ SẴN SÀNG!</h2>
        <p style="font-size: 14.5px; line-height: 1.6; color: #f5deb3;">
            Đường truyền công khai đã được thiết lập qua Cloudflare Tunnel (<b>Không bị hỏi đăng nhập, không lỗi màn hình đen</b>).<br>
            Tài khoản TikTok kết nối: <b style="color: #ffd700;">{clean_username}</b>
        </p>
        <div style="margin: 18px 0; display: flex; flex-wrap: wrap; gap: 12px;">
            <a href="{obs_link}" target="_blank" style="background: linear-gradient(180deg, #d97706 0%, #b45309 100%); color: #fff; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px; border: 1px solid #ffd700; box-shadow: 0 4px 15px rgba(217,119,6,0.4);">
                📺 [1] MỞ MÀN HÌNH LIVE (Dán link này vào TikTok LIVE Studio)
            </a>
            <a href="{admin_link}" target="_blank" style="background: linear-gradient(180deg, #15803d 0%, #166534 100%); color: #fff; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px; border: 1px solid #86efac;">
                🎛️ [2] MỞ TRANG QUẢN TRỊ & THỬ NGHIỆM NÚT BẤM (MOCK PANEL)
            </a>
        </div>
        <div style="background: rgba(0,0,0,0.5); border-left: 4px solid #ffd700; padding: 12px 14px; border-radius: 6px; font-size: 13.5px; color: #ffe4b5;">
            📌 <b>Link dán vào TikTok LIVE Studio / OBS Studio (Nguồn Liên Kết - Kích thước 1080 x 1920):</b><br>
            <code style="color: #00ffcc; background: #000; padding: 5px 10px; border-radius: 4px; display: inline-block; margin-top: 6px; font-size: 14px; word-break: break-all;">{obs_link}</code>
        </div>
    </div>
    """
    display(HTML(html_card))
else:
    print("❌ Không lấy được link Cloudflare. Kiểm tra log:")
    !cat /tmp/cloudflared.log


### 📺 HƯỚNG DẪN CÀI ĐẶT VÀO TIKTOK LIVE STUDIO (CHỈ MẤT 30 GIÂY)
1. Mở phần mềm **TikTok LIVE Studio** trên máy tính của bạn.
2. Bấm vào nút **"Thêm nguồn"** $ightarrow$ chọn mục **"Liên kết"** (biểu tượng mắt xích).
3. Dán link ở mục **[1]** vào ô URL.
4. Đặt độ phân giải là **1080 x 1920** (hoặc chọn Tỉ lệ dọc 9:16).
5. Tích chọn **"Điều khiển âm thanh thông qua TikTok LIVE Studio"** để nghe hiệu ứng rung quẻ, chiêng và giọng đọc TTS.

---
## 🔴 PHẦN 2: TỰ ĐỘNG PHÁT LIVESTREAM LÊN TIKTOK TỪ COLAB (KHÔNG CẦN BẬT MÁY TÍNH)
Phần này sẽ khởi tạo **màn hình ảo 1080x1920** ngay bên trong Google Colab, mở màn hình Game Bốc Quẻ và dùng **FFmpeg** đẩy luồng RTMP thẳng lên TikTok Live qua Stream Key.

In [ ]:
# @title 🎥 BƯỚC 2: Nhập Stream Key & Khởi Chạy Livestream Tự Động 24/7 { display-mode: "form" }
# @markdown Nhập Server URL & Stream Key từ TikTok LIVE Studio / RTMP:

TikTok_Server_URL = "rtmp://live-push.tiktok.com/live/" # @param {type:"string"}
TikTok_Stream_Key = "" # @param {type:"string"}
TikTok_Username = "@vuatiengviet171095" # @param {type:"string"}

import os, time, subprocess

if not TikTok_Stream_Key.strip():
    print("⚠️ VUI LÒNG NHẬP STREAM KEY CỦA TIKTOK VÀO Ô TRÊN TRƯỚC KHI BẤM CHẠY!")
else:
    print("⏳ [1/3] Đang chuẩn bị môi trường màn hình ảo Xvfb & Chrome...")
    !apt-get update -qq > /dev/null 2>&1
    !apt-get install -y -qq xvfb pulseaudio ffmpeg > /dev/null 2>&1
    
    # Dọn dẹp tiến trình phát cũ
    !pkill -9 -f Xvfb > /dev/null 2>&1
    !pkill -9 -f chrome > /dev/null 2>&1
    !pkill -9 -f ffmpeg > /dev/null 2>&1
    time.sleep(1)
    
    # Khởi tạo màn hình ảo 1080x1920 24-bit
    print("📺 Đang khởi tạo Màn hình ảo dọc 1080x1920...")
    subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1080x1920x24"]) 
    os.environ["DISPLAY"] = ":99"
    time.sleep(2)
    
    # Mở trình duyệt Chrome toàn màn hình vào link game
    game_target_url = "http://localhost:3005/index.html"
    print(f"🌐 Đang mở Game Bốc Quẻ trong màn hình ảo: {game_target_url}...")
    chrome_cmd = [
        "google-chrome",
        "--no-sandbox",
        "--disable-dev-shm-usage",
        "--window-size=1080,1920",
        "--start-fullscreen",
        "--kiosk",
        "--autoplay-policy=no-user-gesture-required",
        game_target_url
    ]
    subprocess.Popen(chrome_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    
    # Bắt đầu đẩy luồng RTMP lên TikTok
    rtmp_destination = TikTok_Server_URL.rstrip("/") + "/" + TikTok_Stream_Key.lstrip("/")
    print(f"📡 ĐANG PHÁT TRỰC TIẾP LÊN TIKTOK TẠI: {TikTok_Server_URL}...")
    
    # Tự động nhận diện GPU NVIDIA (T4 trên Colab) để dùng phần cứng h264_nvenc
    has_gpu = (os.system("nvidia-smi > /dev/null 2>&1") == 0 and os.system("ffmpeg -encoders 2>/dev/null | grep -q h264_nvenc") == 0)
    if has_gpu:
        print("🚀 PHÁT HIỆN GPU T4: Tự động kích hoạt bộ tăng tốc phần cứng NVENC (h264_nvenc)!")
        vcodec_opts = "-c:v h264_nvenc -preset p4 -tune ll -b:v 3500k -maxrate 4000k -bufsize 7000k"
    else:
        print("ℹ️ Đang chạy trên CPU: Kích hoạt bộ mã hóa libx264 veryfast...")
        vcodec_opts = "-c:v libx264 -preset veryfast -b:v 3000k -maxrate 3500k -bufsize 6000k"
    
    ffmpeg_cmd = (
        f"ffmpeg -y -f x11grab -draw_mouse 0 -video_size 1080x1920 -framerate 30 -i :99.0 "
        f"-f lavfi -i anullsrc=channel_layout=stereo:sample_rate=44100 "
        f"{vcodec_opts} -pix_fmt yuv420p -g 60 "
        f"-c:a aac -b:a 128k -ar 44100 "
        f'-f flv "{rtmp_destination}" > /tmp/ffmpeg_stream.log 2>&1 &'
    )
    os.system(ffmpeg_cmd)
    time.sleep(3)
    print("🎉 LIVESTREAM TỰ ĐỘNG ĐÃ KHỞI CHẠY THÀNH CÔNG! Kiểm tra trạng thái ở Bước 3 bên dưới.")


In [ ]:
# @title 📊 BƯỚC 3: Kiểm Tra Tốc Độ & Trạng Thái Luồng Phát TikTok
!tail -n 25 /tmp/ffmpeg_stream.log 2>/dev/null || echo "Chưa có log phát sóng."


In [ ]:
# @title ⏹️ BƯỚC 4: Dừng Toàn Bộ Hệ Thống Livestream
!pkill -9 -f ffmpeg
!pkill -9 -f chrome
!pkill -9 -f Xvfb
!pkill -9 -f "server.js"
!pkill -9 -f cloudflared
print("🛑 ĐÃ DỪNG TOÀN BỘ PHIÊN LIVESTREAM!")
